In [ ]:
!pip install timm

In [ ]:
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from tqdm.auto import tqdm
import gc
import matplotlib.pyplot as plt
import numpy as np
import os
import random
import time
import timm # WICHTIG: pip install timm
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.transforms.functional as TF  # Das fixiert den NameError
import re
import cv2 # Für das Resizing der Disparity-Map




# Device Selection (GPU/CPU)
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"Training auf GPU: {torch.cuda.get_device_name(0)}")
else:
    device = torch.device("cpu")
    print("Training auf CPU (Langsam!)")



Training auf GPU: NVIDIA GeForce RTX 3080 Ti


In [ ]:
# --- DATASET V9 (Mit Asymmetric Augmentation) ---
class StereoDataset(Dataset):
    def __init__(self, data_dir, mode='train', use_crop=True, use_augmentation=True):
        self.data_dir = data_dir
        self.mode = mode
        self.use_crop = use_crop
        self.use_augmentation = use_augmentation
        
        # Dateilisten sammeln (angepasst an SceneFlow Struktur)
        self.left_files = []
        self.right_files = []
        self.disp_left_files = []
        self.disp_right_files = []
        
        # Rekursives Suchen nach .pfm Dateien
        for root, dirs, files in os.walk(os.path.join(data_dir, 'frames_finalpass')):
            for file in files:
                if file.endswith('.png') and 'left' in root:
                    l_path = os.path.join(root, file)
                    r_path = l_path.replace('left', 'right')
                    
                    # Disparities finden (Ordnerwechsel frames_finalpass -> disparity)
                    dl_path = l_path.replace('frames_finalpass', 'disparity').replace('.png', '.pfm')
                    dr_path = r_path.replace('frames_finalpass', 'disparity').replace('.png', '.pfm')
                    
                    if os.path.exists(r_path) and os.path.exists(dl_path):
                        self.left_files.append(l_path)
                        self.right_files.append(r_path)
                        self.disp_left_files.append(dl_path)
                        self.disp_right_files.append(dr_path)
                        
        print(f"[{mode.upper()}] {len(self.left_files)} Paare gefunden.")

    def load_pfm(self, file):
        with open(file, "rb") as f:
            header = f.readline().rstrip()
            if header == b'PF': color = True
            elif header == b'Pf': color = False
            else: raise Exception('Not a PFM file.')
            
            dim_match = re.match(rb'^(\d+)\s(\d+)\s$', f.readline())
            if dim_match: width, height = map(int, dim_match.groups())
            else: raise Exception('Malformed PFM header.')
            
            scale = float(f.readline().rstrip())
            if scale < 0: endian = '<' # little endian
            else: endian = '>' # big endian
            
            data = np.fromfile(f, endian + 'f')
            shape = (height, width, 3) if color else (height, width)
            return np.flipud(data.reshape(shape))

    def __len__(self):
        return len(self.left_files)

    def __getitem__(self, idx):
        # 1. Laden
        l_path = self.left_files[idx]
        r_path = self.right_files[idx]
        
        # Grayscale 'L' für MobileNetV3 (1-Channel)
        left = Image.open(l_path).convert('L')
        right = Image.open(r_path).convert('L')
        
        dl = self.load_pfm(self.disp_left_files[idx])
        dr = self.load_pfm(self.disp_right_files[idx])
        
        # Resize auf 640x480 (Training Resolution)
        # Disparität muss mitskaliert werden!
        orig_w, orig_h = left.size
        target_w, target_h = 640, 480
        
        left = left.resize((target_w, target_h), Image.BILINEAR)
        right = right.resize((target_w, target_h), Image.BILINEAR)
        
        scale_x = target_w / orig_w
        scale_y = target_h / orig_h # Nur für Crop relevant, Disp skaliert nur mit Width
        
        # Disp resize (Achtung: Werte skalieren!)
        dl = cv2.resize(dl, (target_w, target_h), interpolation=cv2.INTER_LINEAR) * scale_x
        dr = cv2.resize(dr, (target_w, target_h), interpolation=cv2.INTER_LINEAR) * scale_x

        # Numpy Conversion
        l_np = np.array(left, dtype=np.float32) / 255.0
        r_np = np.array(right, dtype=np.float32) / 255.0
        dl_np = np.ascontiguousarray(dl, dtype=np.float32)
        dr_np = np.ascontiguousarray(dr, dtype=np.float32)

        # 2. Augmentation & Crop
        if self.mode == 'train' and self.use_crop:
            # Random Crop (320x640) - volle Breite behalten!
            crop_h, crop_w = 320, 640
            y = random.randint(0, target_h - crop_h)
            x = random.randint(0, target_w - crop_w)
            
            l_np = l_np[y:y+crop_h, x:x+crop_w]
            r_np = r_np[y:y+crop_h, x:x+crop_w]
            dl_np = dl_np[y:y+crop_h, x:x+crop_w]
            dr_np = dr_np[y:y+crop_h, x:x+crop_w]
            
            if self.use_augmentation:
                # A) ASYMMETRIC PHOTOMETRIC (WICHTIG FÜR SSIM!)
                # Wir verändern Helligkeit/Kontrast für L und R UNABHÄNGIG
                # Das zwingt das Netz, Strukturen statt Helligkeit zu matchen.
                def augment_photo(img):
                    # Random Brightness (0.8 - 1.2)
                    mult = 0.8 + np.random.rand() * 0.4 
                    img = img * mult
                    # Random Contrast
                    img = np.clip(img, 0, 1)
                    mean = img.mean()
                    contrast = 0.8 + np.random.rand() * 0.4
                    img = (img - mean) * contrast + mean
                    # Random Gamma
                    gamma = 0.8 + np.random.rand() * 0.4
                    img = img ** gamma
                    return np.clip(img, 0, 1)

                l_np = augment_photo(l_np)
                r_np = augment_photo(r_np) # Anders als links!

        # To Tensor
        l_t = torch.from_numpy(l_np).unsqueeze(0) # [1, H, W]
        r_t = torch.from_numpy(r_np).unsqueeze(0)
        dl_t = torch.from_numpy(dl_np).unsqueeze(0)
        dr_t = torch.from_numpy(dr_np).unsqueeze(0)
        
        return l_t, r_t, dl_t, dr_t

In [ ]:
# --- NPU-FRIENDLY ARCHITECTURE V9 ---

class StructureBlock(nn.Module):
    """
    Berechnet Sobel-Kanten und Laplacian On-the-Fly auf der NPU.
    Inferenz-Input bleiben reine Bilder, keine Vorverarbeitung nötig!
    """
    def __init__(self):
        super().__init__()
        # Sobel X
        sobel_x = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]]).view(1, 1, 3, 3)
        # Sobel Y
        sobel_y = torch.tensor([[-1., -2., -1.], [0., 0., 0.], [1., 2., 1.]]).view(1, 1, 3, 3)
        # Laplacian
        laplace = torch.tensor([[0., 1., 0.], [1., -4., 1.], [0., 1., 0.]]).view(1, 1, 3, 3)
        
        # Als konstante Gewichte registrieren (kein Training, NPU optimiert dies weg)
        self.register_buffer('k_sx', sobel_x)
        self.register_buffer('k_sy', sobel_y)
        self.register_buffer('k_lap', laplace)

    def forward(self, x):
        # x: [B, 1, H, W]
        sx = F.conv2d(x, self.k_sx, padding=1)
        sy = F.conv2d(x, self.k_sy, padding=1)
        lap = F.conv2d(x, self.k_lap, padding=1)
        
        # Magnitude berechnen für Sobel
        mag = torch.sqrt(sx*sx + sy*sy + 1e-6)
        
        # Return: [B, 2, H, W] -> (SobelMag, Laplace)
        return torch.cat([mag, lap], dim=1)

class Conv2dReLU6(nn.Module):
    """ Standard Block für Hailo (ReLU6 ist Quantisierungs-freundlich) """
    def __init__(self, in_c, out_c, kernel_size=3, stride=1, padding=1):
        super().__init__()
        self.conv = nn.Conv2d(in_c, out_c, kernel_size, stride, padding, bias=False)
        self.bn = nn.BatchNorm2d(out_c)
        self.act = nn.ReLU6(inplace=True)

    def forward(self, x):
        return self.act(self.bn(self.conv(x)))

class DepthwiseSeparable(nn.Module):
    """ Effizienter Block für Mini U-Net """
    def __init__(self, in_c, out_c):
        super().__init__()
        self.depthwise = nn.Conv2d(in_c, in_c, 3, padding=1, groups=in_c, bias=False)
        self.pointwise = nn.Conv2d(in_c, out_c, 1, bias=False)
        self.bn = nn.BatchNorm2d(out_c)
        self.act = nn.ReLU6(inplace=True)
    
    def forward(self, x):
        x = self.depthwise(x)
        x = self.act(self.bn(self.pointwise(x)))
        return x

class MiniUNetRefiner(nn.Module):
    """ 
    Ersetzt die einfachen ResBlocks. 
    Sieht Kontext durch Downsampling und nutzt Structure-Guidance.
    """
    def __init__(self, in_channels):
        super().__init__()
        # Input: Disp(1) + Feature(32) + Structure(2) = 35 channels
        
        # Encoder
        self.enc1 = Conv2dReLU6(in_channels, 32)
        self.pool1 = nn.MaxPool2d(2) # /2
        
        self.enc2 = Conv2dReLU6(32, 64)
        self.pool2 = nn.MaxPool2d(2) # /4
        
        # Bottleneck (Dilated für Context)
        self.center = nn.Sequential(
            Conv2dReLU6(64, 64),
            DepthwiseSeparable(64, 64)
        )
        
        # Decoder
        self.up2 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.dec2 = Conv2dReLU6(64 + 64, 32) # Skip Connection von enc2
        
        self.up1 = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)
        self.dec1 = Conv2dReLU6(32 + 32, 16) # Skip von enc1
        
        # Final Output (Residual Correction)
        self.final = nn.Conv2d(16, 1, kernel_size=3, padding=1)
        # Weight init klein halten
        nn.init.uniform_(self.final.weight, -0.01, 0.01)

    def forward(self, disp_curr, features, structure):
        # Concatenate Inputs
        x = torch.cat([disp_curr, features, structure], dim=1)
        
        # U-Net Flow
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool1(e1))
        
        c = self.center(self.pool2(e2))
        
        d2 = self.dec2(torch.cat([self.up2(c), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        
        res = self.final(d1)
        return F.relu(disp_curr + res) # ReLU garantiert positive Disparität

class StereoNet_NPU_V9(nn.Module):
    def __init__(self, max_disp=192):
        super().__init__()
        self.max_disp = max_disp
        
        # --- 1. SHARED BACKBONE (MobileNetV3 Large) ---
        # features_only=True gibt uns Feature Maps verschiedener Strides
        # out_indices=(1, 2, 4) -> Stride 4, 8, 16 (typisch für MNV3)
        self.backbone = timm.create_model('mobilenetv3_large_100', 
                                          pretrained=True, 
                                          features_only=True, 
                                          out_indices=(1, 2, 4),
                                          in_chans=1) # Wir nutzen Grayscale
        
        # Feature Adapter: Stride 4 Output von MNV3 (24 channels) auf 32 für Cost Volume
        self.adapter_s4 = nn.Conv2d(24, 32, kernel_size=1, bias=False)
        
        # --- 2. STRUCTURE BLOCK (Kanten) ---
        self.structure = StructureBlock()
        
        # --- 3. COST VOLUME & MATCHING ---
        self.cost_conv = nn.Sequential(
            DepthwiseSeparable(32, 32),
            DepthwiseSeparable(32, 32),
            nn.Conv2d(32, 1, 1, bias=False)
        )
        
        # --- 4. REFINEMENT PYRAMID (Mini U-Nets) ---
        # Input Channels: 1(Disp) + 32(Feat) + 2(Struct) = 35
        self.refine_low = MiniUNetRefiner(35) # 1/4 Scale
        self.refine_v1  = MiniUNetRefiner(35) # 1/2 Scale
        self.refine_final = MiniUNetRefiner(35) # Full Scale
        
        # --- 5. OCCLUSION HEAD ---
        # Nutzt Features aus dem Low-Res Refiner + Disparität
        # Input: 1 Channel (Disp) + 2 Channels (Structure: Sobel+Laplace) = 3
        self.occ_head = nn.Sequential(Conv2dReLU6(3, 16),
                                    nn.Conv2d(16, 1, 1))

        
    def forward_single(self, left, right):
        # 1. Feature Extraction (Backbone)
        # MNV3 liefert Liste: [Stride4, Stride8, Stride16]
        feats_l_all = self.backbone(left)
        feats_r_all = self.backbone(right)
        
        # Wir nehmen nur Stride 4 (Index 0 in unserer Liste)
        feat_l = self.adapter_s4(feats_l_all[0])
        feat_r = self.adapter_s4(feats_r_all[0])
        
        # 2. Structure Extraction (Parallel)
        struct_l = self.structure(left) # [B, 2, H, W]
        
        # 3. Cost Volume (Low Res 1/8 intern, output 1/8)
        # Wir machen einfaches Correlation Volume anstatt 4D concat (RAM sparen)
        # Aber hier bleiben wir beim bewährten "Feature Difference" Ansatz für NPU
        # Um VRAM zu sparen, berechnen wir Cost Vol auf 1/8 Scale (Stride 2 auf Feats)
        
        # Downsample features auf 1/8 für Cost Vol
        f_l_8 = F.avg_pool2d(feat_l, 2)
        f_r_8 = F.avg_pool2d(feat_r, 2)
        
        B, C, H8, W8 = f_l_8.shape
        max_disp_8 = self.max_disp // 8
        
        cost_vol = []
        for d in range(max_disp_8):
            if d > 0:
                shifted = F.pad(f_r_8[:, :, :, :-d], (d, 0, 0, 0))
            else:
                shifted = f_r_8
            diff = torch.abs(f_l_8 - shifted)
            cost_vol.append(diff)
        cost_vol = torch.stack(cost_vol, dim=1) # [B, D, C, H, W]
        
        # Cost Aggregation
        # [B, D, C, H, W] -> [B*D, C, H, W] für Conv
        cost_vol = cost_vol.view(B*max_disp_8, C, H8, W8)
        cost_out = self.cost_conv(cost_vol) # -> [B*D, 1, H, W]
        cost_out = cost_out.view(B, max_disp_8, H8, W8)
        
        # Soft Argmax
        prob = F.softmax(-cost_out, dim=1)
        d_range = torch.arange(max_disp_8, device=left.device).view(1, -1, 1, 1).float()
        disp_8 = torch.sum(prob * d_range, dim=1, keepdim=True) # [B, 1, H, W]
        
        # 4. Refinement Cascade
        
        # --- Stage Low (1/4 Scale) ---
        # Upsample 1/8 -> 1/4
        disp_4 = F.interpolate(disp_8 * 2.0, scale_factor=2, mode='bilinear')
        # Structure und Features auch auf 1/4 (Features sind schon da)
        struct_4 = F.interpolate(struct_l, scale_factor=0.25, mode='bilinear')
        
        disp_low_ref = self.refine_low(disp_4, feat_l, struct_4)
        
        # --- Stage V1 (1/2 Scale) ---
        disp_2 = F.interpolate(disp_low_ref * 2.0, scale_factor=2, mode='bilinear')
        feat_2 = F.interpolate(feat_l, scale_factor=2, mode='bilinear')
        struct_2 = F.interpolate(struct_l, scale_factor=0.5, mode='bilinear')
        
        disp_v1_ref = self.refine_v1(disp_2, feat_2, struct_2)
        
        # --- Stage Final (Full Scale) ---
        disp_1 = F.interpolate(disp_v1_ref * 2.0, scale_factor=2, mode='bilinear')
        feat_1 = F.interpolate(feat_l, scale_factor=4, mode='bilinear') # Teuer, aber nötig für U-Net Input
        
        disp_final = self.refine_final(disp_1, feat_1, struct_l)
        
        # --- Occ Head ---
        # OccHead bekommt [Disp, Sobel] -> Lightweight
        # Wir geben dem OccHead die Disparität und die Kanteninfos
        occ_logits = self.occ_head(torch.cat([disp_final, struct_l], dim=1))
        return disp_final, disp_v1_ref, disp_low_ref, occ_logits

    def core(self, left, right):
        # Wrapper für Training/Inferenz Einheitlichkeit
        return self.forward_single(left, right)

# Fix OccHead im Init oben entsprechend anpassen:
# self.occ_head = nn.Sequential(Conv2dReLU6(3, 16), nn.Conv2d(16, 1, 1)) # Input: Disp(1) + Struct(2)
# Und im Forward: occ_logits = self.occ_head(torch.cat([disp_final, struct_l], dim=1))

In [4]:
@torch.no_grad()
def validate(model, val_loader, device):
    model.eval()
    
    total_epe = 0.0
    total_loss = 0.0
    valid_batches = 0
    
    # tqdm für Fortschrittsbalken
    pbar = tqdm(val_loader, desc="🔍 Validierung", leave=False, ncols=150)
    
    # FIX: Jetzt 4 Werte entpacken statt 3
    for left, right, gt_L, gt_R in pbar:
        left, right = left.to(device), right.to(device)
        gt_L = gt_L.to(device)
        # gt_R brauchen wir für EPE-Validierung eigentlich nicht zwingend, 
        # aber wir müssen es entpacken, damit Python nicht meckert.

        # Forward Pass (Wir nutzen nur LR Core für Speed)
        # model.core gibt zurück: (disp_final, disp_v1, disp_low, occ)
        out = model.core(left, right)
        disp_pred = out[0] # Wir nehmen nur die finale Disparität
        
        # Validitäts-Maske (Nur Pixel prüfen, die Ground Truth haben)
        mask = (gt_L > 0) & (gt_L < 192)
        
        if mask.sum() > 0:
            # 1. EPE (End Point Error) berechnen
            # Absoluter Abstand in Pixeln
            diff = torch.abs(disp_pred[mask] - gt_L[mask])
            epe = diff.mean().item()
            
            # 2. Loss berechnen (Smooth L1 als Referenz)
            loss = F.smooth_l1_loss(disp_pred[mask], gt_L[mask], beta=1.0).item()
            
            total_epe += epe
            total_loss += loss
            valid_batches += 1
            
            pbar.set_postfix({'val_epe': f"{epe:.2f}"})

    if valid_batches == 0:
        return 0.0, 0.0

    return (total_loss / valid_batches), (total_epe / valid_batches)


In [ ]:


# --- 1. SSIM (Structural Similarity) ---
def get_ssim_window(window_size, channel):
    def gaussian(window_size, sigma):
        gauss = torch.Tensor([np.exp(-(x - window_size//2)**2/float(2*sigma**2)) for x in range(window_size)])
        return gauss/gauss.sum()
    
    _1D_window = gaussian(window_size, 1.5).unsqueeze(1)
    _2D_window = _1D_window.mm(_1D_window.t()).float().unsqueeze(0).unsqueeze(0)
    window = _2D_window.expand(channel, 1, window_size, window_size).contiguous()
    return window

class SSIM(nn.Module):
    def __init__(self, window_size=11, channel=1):
        super(SSIM, self).__init__()
        self.window_size = window_size
        self.channel = channel
        self.register_buffer('window', get_ssim_window(window_size, channel))

    def forward(self, img1, img2):
        mu1 = F.conv2d(img1, self.window, padding=self.window_size//2, groups=self.channel)
        mu2 = F.conv2d(img2, self.window, padding=self.window_size//2, groups=self.channel)

        mu1_sq = mu1.pow(2)
        mu2_sq = mu2.pow(2)
        mu1_mu2 = mu1*mu2

        sigma1_sq = F.conv2d(img1*img1, self.window, padding=self.window_size//2, groups=self.channel) - mu1_sq
        sigma2_sq = F.conv2d(img2*img2, self.window, padding=self.window_size//2, groups=self.channel) - mu2_sq
        sigma12 = F.conv2d(img1*img2, self.window, padding=self.window_size//2, groups=self.channel) - mu1_mu2

        C1 = 0.01**2
        C2 = 0.03**2

        ssim_map = ((2*mu1_mu2 + C1)*(2*sigma12 + C2))/((mu1_sq + mu2_sq + C1)*(sigma1_sq + sigma2_sq + C2))
        return ssim_map.mean()

# --- 2. Charbonnier Loss (Robuste L1 Variante) ---
def charbonnier_loss(x, y, eps=1e-3):
    diff = x - y
    loss = torch.sqrt(diff * diff + eps * eps)
    return loss.mean()

# --- 3. Smoothness (1st + 2nd Order) ---
def smoothness_loss_2nd(pred_disp, img, beta=12.0):
    def gradient(x):
        h_x = x.size()[-2]
        w_x = x.size()[-1]
        r = F.pad(x, (0, 1, 0, 0))[:, :, :, 1:] - x
        l = x - F.pad(x, (1, 0, 0, 0))[:, :, :, :-1]
        d = F.pad(x, (0, 0, 0, 1))[:, :, 1:, :] - x
        u = x - F.pad(x, (0, 0, 1, 0))[:, :, :-1, :]
        return r, l, d, u

    grad_disp_r, grad_disp_l, grad_disp_d, grad_disp_u = gradient(pred_disp)
    grad_img_r, _, grad_img_d, _ = gradient(img) # Nur R/D für Gewichtung nötig

    weight_x = torch.exp(-torch.abs(grad_img_r) * beta)
    weight_y = torch.exp(-torch.abs(grad_img_d) * beta)

    # 1st Order Smoothness
    smooth1 = (torch.abs(grad_disp_r) * weight_x).mean() + (torch.abs(grad_disp_d) * weight_y).mean()
    
    # 2nd Order (Krümmung)
    grad2_x = torch.abs(grad_disp_r - grad_disp_l)
    grad2_y = torch.abs(grad_disp_d - grad_disp_u)
    smooth2 = (grad2_x * weight_x).mean() + (grad2_y * weight_y).mean()

    return smooth1 + 0.5 * smooth2

# --- 4. ROBUST STEREO LOSS V9 (Combined) ---
def robust_stereo_loss_v9(outputs, left_img, right_img, gt_disp_L, 
                          w_geom=0.6, w_photo=1.0, w_lrc=0.6, w_smooth=0.1, w_occ=0.2):
    
    # Initialisiere SSIM Modul (wird gecached)
    if not hasattr(robust_stereo_loss_v9, 'ssim_module'):
        robust_stereo_loss_v9.ssim_module = SSIM(channel=1).to(left_img.device)
    ssim_loss_fn = robust_stereo_loss_v9.ssim_module

    # Multi-Scale Weights: [Final, V1, Low]
    scale_weights = [1.0, 0.5, 0.25]
    
    total_loss = 0.0
    logs = {}

    # Inputs für Warping vorbereiten
    B, _, H, W = left_img.shape
    
    # Grid Construction (Einmalig)
    grid_y, grid_x = torch.meshgrid(torch.arange(H, device=left_img.device), torch.arange(W, device=left_img.device), indexing='ij')
    grid_base = torch.stack((grid_x, grid_y), dim=0).unsqueeze(0).expand(B, -1, -1, -1).float() # [B, 2, H, W]

    # --- LOOP ÜBER SKALEN ---
    for i, weight in enumerate(scale_weights):
        # Disparities holen
        disp_L = outputs["LR"][i]
        
        # RL Pass existiert? (Für LRC Check)
        disp_R = outputs["RL"][i] if "RL" in outputs else None
        
        # --- 1. SUPERVISED GEOMETRY ---
        if gt_disp_L is not None:
            mask_valid = (gt_disp_L > 0) & (gt_disp_L < 192)
            # Charbonnier statt SmoothL1 für Robustheit
            loss_g = charbonnier_loss(disp_L[mask_valid], gt_disp_L[mask_valid])
            total_loss += w_geom * weight * loss_g
            if i==0: logs['geom'] = loss_g.item()

        # --- 2. LRC & OCCLUSION MASK (Ohne GT!) ---
        if disp_R is not None:
            # Warp R nach L (Check Consistency)
            # Normierung für grid_sample
            disp_L_norm = 2.0 * disp_L / (W - 1) 
            grid_norm_x = (2.0 * grid_x / (W - 1)) - 1.0
            grid_norm_y = (2.0 * grid_y / (H - 1)) - 1.0
            
            # Projektion: Wo landet Pixel x wenn ich disp_L anwende?
            # grid hat shape [B, 2, H, W]. Wir modifizieren X-Kanal.
            # Grid muss [B, H, W, 2] sein für grid_sample
            vgrid = torch.stack((grid_norm_x - disp_L_norm.squeeze(1), grid_norm_y.expand(B,H,W)), dim=3)
            
            # Hole den Disparitätswert von rechts an der Stelle, wo links hinzeigt
            disp_R_warped = F.grid_sample(disp_R, vgrid, align_corners=True, padding_mode='border')
            
            # Konsistenz-Check: |dL - dR_warped|
            lrc_diff = torch.abs(disp_L - disp_R_warped)
            loss_l = charbonnier_loss(lrc_diff, torch.zeros_like(lrc_diff)) # Minimiere Differenz
            total_loss += w_lrc * weight * loss_l
            if i==0: logs['lrc'] = loss_l.item()

            # Berechne Visibility Maske (Alles was konsistent ist = sichtbar)
            # Toleranz: 1.0 Pixel + 0.01 * disparity (relative Toleranz)
            mask_vis = (lrc_diff < (1.0 + 0.01 * disp_L)).float().detach()
            
            # --- 3. PHOTOMETRIC LOSS (SSIM + Charbonnier) ---
            # Warpe das rechte Bild nach links
            right_img_warped = F.grid_sample(right_img, vgrid, align_corners=True, padding_mode='border')
            
            # SSIM (gewicht 0.85)
            # SSIM liefert 1.0 für perfekte Matches -> Loss ist 1 - SSIM
            # Maskierung ist bei SSIM tricky, wir machen einfaches element-wise Weighting approximation
            # Da unsere SSIM Implementierung global mittelt, nutzen wir hier L1+SSIM Mix auf Pixelbasis
            
            # Einfacher: L1 Charbonnier Photo Loss (auf maskierten Pixeln)
            photo_diff = charbonnier_loss(left_img, right_img_warped) # Pixel-wise distance map theoretisch
            # Aber charbonnier_loss returned mean. Wir rechnen manuell:
            diff_map = torch.sqrt((left_img - right_img_warped)**2 + 1e-6)
            loss_p_charb = (diff_map * mask_vis).sum() / (mask_vis.sum() + 1e-6)
            
            # SSIM (Global berechnet, aber wir vertrauen darauf dass SSIM Strukturen besser matcht)
            # SSIM ignoriert Masken im Standard-Code. Workaround:
            # Wir nutzen SSIM als globalen Guide.
            s_val = ssim_loss_fn(left_img * mask_vis, right_img_warped * mask_vis)
            loss_p_ssim = 1.0 - s_val
            
            # Kombi: 0.85 SSIM + 0.15 L1
            loss_photo = 0.85 * loss_p_ssim + 0.15 * loss_p_charb
            total_loss += w_photo * weight * loss_photo
            if i==0: logs['photo'] = loss_photo.item()

            # --- 4. OCC HEAD TRAINING ---
            # Trainiere den OCC Head (Index 3) gegen (1 - mask_vis)
            if i == 0: # Nur auf feinster Stufe
                occ_pred = outputs["LR"][3] # Logits
                occ_target = 1.0 - mask_vis
                loss_o = F.binary_cross_entropy_with_logits(occ_pred, occ_target)
                total_loss += w_occ * loss_o
                logs['occ'] = loss_o.item()
                
    # 5. Smoothness (Nur auf Final Scale)
    loss_s = smoothness_loss_2nd(outputs["LR"][0], left_img)
    total_loss += w_smooth * loss_s
    
    return total_loss, logs





def train_strategic_v9(
    model, 
    base_dir, 
    # --- Training Hyperparameter ---
    epochs=75, 
    lr_max=2e-4, 
    weight_decay=1e-5,
    warmup_pct=0.1,      # Prozent der Epochen für Warmup
    grad_clip=1.0,
    
    # --- Dataloader & Dataset ---
    batch_size=6, 
    num_workers=4, 
    use_crop=True,       # Training mit Random Crop?
    use_aug=True        # Training mit Color Augmentation?
):
    
    # 1. Datasets & Loader (Voll parametrisiert)
    train_ds = StereoDataset(base_dir, mode='train', use_crop=use_crop, use_augmentation=use_aug)
    val_ds = StereoDataset(base_dir, mode='val', use_crop=False, use_augmentation=False)
    
    train_loader = DataLoader(
        train_ds, 
        batch_size=batch_size, 
        shuffle=True, 
        num_workers=num_workers, 
        pin_memory=True, 
        persistent_workers=True,
        prefetch_factor=2
    )
    
    val_loader = DataLoader(
        val_ds, 
        batch_size=1, 
        shuffle=False, 
        num_workers=2,
        pin_memory=True
    )

    
    # Optimizer (Backbone hat pre-trained weights, LR evtl etwas kleiner dort?)
    # Wir nutzen einheitliche LR für Simplicity, AdamW regelt das
    optimizer = optim.AdamW(model.parameters(), lr=lr_max, weight_decay=weight_decay)
    
    scheduler = optim.lr_scheduler.OneCycleLR(
        optimizer, max_lr=lr_max, total_steps=epochs * len(train_loader),
        pct_start=warmup_pct, div_factor=25, final_div_factor=1000
    )
    
    scaler = torch.cuda.amp.GradScaler()
    current_best_epe = float('inf')
    
    print(f"🚀 V9 TRAINING START | Backbone: MobileNetV3-Large | BS={batch_size}")
    
    # Log Header
    log_file = "training_log_FusedBackbone-Stereo.txt"
    with open(log_file, "w") as f:
        f.write("epoch\tavg_loss\tval_epe\tgeom\tphoto\tlrc\tocc\tlr\n")

    for epoch in range(epochs):
        model.train()
        sum_logs = {"geom": 0.0, "photo": 0.0, "lrc": 0.0, "occ": 0.0}
        epoch_loss = 0.0
        
        pbar = tqdm(train_loader, desc=f"Ep {epoch+1}", ncols=120)
        
        for batch_idx, (l, r, dl, dr) in enumerate(pbar):
            l, r = l.to(device), r.to(device)
            dl, dr = dl.to(device), dr.to(device)
            curr_lr = scheduler.get_last_lr()[0]
            with torch.cuda.amp.autocast():
                # 1. Forward Pass LR
                # Returns: disp_final, disp_v1, disp_low, occ
                out_LR = model.core(l, r) 
                
                # 2. Forward Pass RL (Siamese, flipped inputs)
                l_flip, r_flip = torch.flip(l, [3]), torch.flip(r, [3])
                out_RL_raw = model.core(r_flip, l_flip)
                # Flip outputs back
                out_RL = tuple(torch.flip(o, [3]) for o in out_RL_raw)
                
                # Pack outputs
                outputs = {"LR": out_LR, "RL": out_RL}
                
                # 3. Calculate Loss V9
                loss, logs = robust_stereo_loss_v9(
                    outputs, l, r, dl, dr,
                    w_geom=0.6, w_photo=1.0, w_lrc=0.6, 
                    w_smooth=0.1, w_occ=0.2
                )
                
                epoch_loss += loss.item()
                for k, v in logs.items(): sum_logs[k] += v

            optimizer.zero_grad()
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            
            vram_gb = torch.cuda.memory_reserved(device) / 1024**3
            pbar.set_postfix({
                'VRAM': f"{vram_gb:.1f}G",
                'Loss': f"{loss.item():.2f}",
                'Geom': f"{logs.get('geom', 0):.2f}",     # Sicherer Zugriff
                'Photo': f"{logs.get('photo', 0):.2f}",   # Korrekter Schlüssel & sicherer Zugriff
                'LRC': f"{logs.get('lrc', 0):.2f}",       # LRC ist auch ein nützlicher Wert
                'LR': f"{curr_lr:.1e}",
                'Grad': f"{grad_norm:.1f}"
            })

        # Validation & Logging (gekürzt)
        val_loss, val_epe = validate(model, val_loader, device)
        avg_loss = epoch_loss / len(train_loader)
        avg_logs = {k: v / len(train_loader) for k, v in sum_logs.items()}
        
        print(f"📈 Ep {epoch+1}: Val-EPE: {val_epe:.2f} px")
        
        final_lr = scheduler.get_last_lr()[0]
        with open(log_file, "a") as f:
            f.write(f"{epoch+1}\t{avg_loss:.4f}\t{val_epe:.4f}\t"
                    f"{avg_logs.get('geom', 0):.4f}\t"
                    f"{avg_logs.get('photo', 0):.4f}\t"
                    f"{avg_logs.get('lrc', 0):.4f}\t"
                    f"{avg_logs.get('occ', 0):.4f}\t" # Occ war im Log vergessen
                    f"{final_lr:.2e}\n") # Nur relevante Werte loggen

        # Visuals
        save_debug_visuals(model, val_loader.dataset, epoch+1, index=6)
        plot_disparity_profile(model, val_loader.dataset, epoch+1, index=6)
        save_preview(model, val_loader.dataset, f"epoch_{(epoch+1):03d}", index=6)
        save_symmetry_comparison(model, val_loader.dataset, device, epoch=epoch+1, index=6)

        # Save Checkpoints
        if val_epe < current_best_epe:
            current_best_epe = val_epe
            torch.save(model.state_dict(), "FusedBackbone-Stereo_BEST.pth")
        if (epoch + 1) % 5 == 0:
            torch.save(model.state_dict(), f"FusedBackbone-Stereo_ep_{epoch+1}.pth")
            
    print("✅ V9 Training Complete.")

     




/home/slarc/miniconda3/envs/stereo_wsl/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
def save_debug_visuals(model, dataset, epoch, index=6):
    model.eval()
    
    # 1. Daten holen & Vorbereiten
    l, r, dl, dr = dataset[index]
    l_in = l.unsqueeze(0).to(device)
    r_in = r.unsqueeze(0).to(device)
    
    with torch.no_grad():
        # Core liefert 4 Werte: Final, V1, Low, Occ-Logits
        disp_final, disp_v1, disp_low, _ = model.core(l_in, r_in)
        
    # Hilfsfunktion zum sauberen Konvertieren für Matplotlib (Grayscale-Safe)
    def to_img_np(t):
        arr = t.squeeze().cpu().numpy()
        return arr # Bei Grayscale einfach [H, W] zurückgeben

    fig, axs = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle(f"Debug Visuals - Epoche {epoch}", fontsize=16)

    # Zeile 1: Input & GT & Final Prediction
    axs[0,0].imshow(to_img_np(l), cmap='gray')
    axs[0,0].set_title("Input Image (Left)")
    
    axs[0,1].imshow(to_img_np(dl), cmap='magma', vmin=0, vmax=192)
    axs[0,1].set_title("Ground Truth Disparity")
    
    im_final = axs[0,2].imshow(to_img_np(disp_final), cmap='magma', vmin=0, vmax=192)
    axs[0,2].set_title("Final Prediction (1/1)")
    fig.colorbar(im_final, ax=axs[0,2])

    # Zeile 2: Die Pyramiden-Stufen
    axs[1,0].imshow(to_img_np(disp_low), cmap='magma', vmin=0, vmax=192)
    axs[1,0].set_title("Stage Low (1/4 Scale)")
    
    axs[1,1].imshow(to_img_np(disp_v1), cmap='magma', vmin=0, vmax=192)
    axs[1,1].set_title("Stage V1 (1/2 Scale)")
    
    # EPE Map (Fehlerbild)
    gt_mask = (dl > 0) & (dl < 192)
    epe_map = np.abs(to_img_np(dl) - to_img_np(disp_final))
    epe_map[~gt_mask.squeeze().numpy()] = 0 
    im_epe = axs[1,2].imshow(epe_map, cmap='jet', vmin=0, vmax=10)
    axs[1,2].set_title("EPE Map (Error)")
    fig.colorbar(im_epe, ax=axs[1,2])

    for ax in axs.flatten(): ax.axis('off')
    plt.tight_layout()
    
    os.makedirs("training_progress", exist_ok=True)
    plt.savefig(f"training_progress/FusedBackbone-Stereo_debug_ep_{epoch:03d}.png")
    plt.close()


def save_preview(model, dataset, name, index=6):
    model.eval()
    
    l, r, dl, dr = dataset[index]
    l_in = l.unsqueeze(0).to(device)
    r_in = r.unsqueeze(0).to(device)
    
    with torch.no_grad():
        # A) LR Pass
        disp_LR, _, _, occ_logits = model.core(l_in, r_in)
        
        # B) RL Pass (für Symmetrie-Check / Echo)
        l_f, r_f = torch.flip(l_in, [3]), torch.flip(r_in, [3])
        disp_RL_f, _, _, _ = model.core(r_f, l_f)
        disp_RL = torch.flip(disp_RL_f, [3])
        
        # C) Echo-Map Grid (LRC Check)
        B, _, H, W = disp_LR.shape
        grid_x = torch.arange(W, device=device).view(1, 1, 1, W).expand(B, 1, H, W).float()
        x_proj = grid_x - disp_LR
        grid_y = torch.arange(H, device=device).view(1, 1, H, 1).expand(B, 1, H, W).float()
        norm_x = 2.0 * x_proj / (W - 1) - 1.0
        norm_y = 2.0 * grid_y / (H - 1) - 1.0
        grid = torch.stack((norm_x.squeeze(1), norm_y.squeeze(1)), dim=3)
        
        disp_RL_warped = F.grid_sample(disp_RL, grid, align_corners=False, padding_mode='border')
        echo_map = torch.abs(disp_LR - disp_RL_warped)
        
        # D) Okklusions-Wahrscheinlichkeit (Sigmoid)
        occ_prob = torch.sigmoid(occ_logits)

    # Hilfsfunktion für NP-Konvertierung (Grayscale)
    def to_np(t): return t.squeeze().cpu().numpy()
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 10))
    fig.suptitle(f"Preview Analysis: {name} (Robot Logic)", fontsize=16)

    # 1. Input
    axes[0, 0].imshow(to_np(l), cmap='gray')
    axes[0, 0].set_title("Input Left")
    
    # 2. GT
    axes[0, 1].imshow(to_np(dl), cmap='magma', vmin=0, vmax=192)
    axes[0, 1].set_title("Ground Truth")
    
    # 3. Vorhersage Disparität
    im_lr = axes[0, 2].imshow(to_np(disp_LR), cmap='magma', vmin=0, vmax=192)
    axes[0, 2].set_title("Predicted Disparity")
    fig.colorbar(im_lr, ax=axes[0, 2])

    # 4. Echo-Map (Wo widersprechen sich LR und RL?)
    im_echo = axes[1, 0].imshow(to_np(echo_map), cmap='hot', vmin=0, vmax=5)
    axes[1, 0].set_title("Echo-Map (LRC Error)")
    fig.colorbar(im_echo, ax=axes[1, 0])

    # 5. NEU: Predicted Occlusion (Was der Roboter "sieht")
    # Weiß = Verdeckt/Ungültig, Schwarz = Sicher/Sichtbar
    im_occ = axes[1, 1].imshow(to_np(occ_prob), cmap='gray', vmin=0, vmax=1)
    axes[1, 1].set_title("Predicted Occlusion Head")
    fig.colorbar(im_occ, ax=axes[1, 1])
    
    # 6. EPE Map
    gt_mask = (dl > 0) & (dl < 192)
    epe_map = np.abs(to_np(dl) - to_np(disp_LR))
    epe_map[~gt_mask.squeeze().numpy()] = 0 
    im_epe = axes[1, 2].imshow(epe_map, cmap='jet', vmin=0, vmax=10)
    axes[1, 2].set_title("EPE Error Map")
    fig.colorbar(im_epe, ax=axes[1, 2])
    
    for ax in axes.flatten(): ax.axis('off')
    plt.tight_layout()
    
    os.makedirs("training_progress", exist_ok=True)
    plt.savefig(f"training_progress/FusedBackbone-Stereo_preview_{name}.png", bbox_inches='tight')
    plt.close(fig)


def plot_disparity_profile(model, dataset, epoch, index=0):
    model.eval()
    
    l, r, dl, dr = dataset[index]
    l_in = l.unsqueeze(0).to(device)
    r_in = r.unsqueeze(0).to(device)
    
    with torch.no_grad():
        # Unpacking von 4 Werten
        disp_final, disp_v1, disp_low, _ = model.core(l_in, r_in)
        
    # Helper: Squeezed auf [H, W]
    def to_np(t): return t.squeeze().cpu().numpy()
    
    # FIX: Nur transponieren, wenn es 3 Kanäle hat. Bei [H, W] direkt nutzen.
    left_img = to_np(l)
    if left_img.ndim == 3: # Falls [3, H, W]
        left_img = left_img.transpose(1, 2, 0)
    
    gt_np = to_np(dl)
    final_np = to_np(disp_final)
    v1_np = to_np(disp_v1)
    
    H, W = final_np.shape
    rows = [int(H * 0.25), int(H * 0.50), int(H * 0.75)]
    labels = ["25%", "50%", "75%"]
    
    fig, axs = plt.subplots(4, 1, figsize=(12, 16))
    
    # Bild mit Linien (cmap='gray' hinzugefügt)
    axs[0].imshow(left_img, cmap='gray' if left_img.ndim == 2 else None)
    for row in rows: axs[0].axhline(row, color='yellow', linewidth=2)
    axs[0].set_title(f"Profile Lines (Epoch {epoch})")
    axs[0].axis('off')
    
    # Profile
    for i, (row, label) in enumerate(zip(rows, labels)):
        ax = axs[i+1]
        ax.plot(gt_np[row, :], 'k-', label='Ground Truth', linewidth=2)
        ax.plot(final_np[row, :], 'r-', label='Final Prediction', alpha=0.8)
        ax.plot(v1_np[row, :], 'g--', label='V1 Coarse', alpha=0.6)
        
        ax.set_title(f"Profile at {label} Height")
        ax.set_ylim(-5, 200)
        ax.grid(True, alpha=0.3)
        if i==0: ax.legend()
        
    plt.tight_layout()
    os.makedirs("training_progress", exist_ok=True)
    plt.savefig(f"training_progress/FusedBackbone-Stereo_profile_ep{epoch:03d}.png")
    plt.close(fig)

def save_symmetry_comparison(model, dataset, device, epoch=0, index=0):
    model.eval()
    l, r, dl, dr = dataset[index]
    l_in, r_in = l.unsqueeze(0).to(device), r.unsqueeze(0).to(device)
    
    with torch.no_grad():
        # FIX 1: Variable hieß vorher disp_RL_f, muss aber disp_LR sein
        # FIX 2: Unpacking von 4 Werten
        disp_LR, _, _, _ = model.core(l_in, r_in)
        
        # RL Pass (Inputs flippen)
        l_f, r_f = torch.flip(l_in, [3]), torch.flip(r_in, [3])
        # FIX 3: Auch hier 4 Werte entpacken
        disp_RL_f, _, _, _ = model.core(r_f, l_f)
        disp_RL = torch.flip(disp_RL_f, [3]) 
        
        # LRC Check (Warp RL to Left)
        B, _, H, W = disp_LR.shape
        grid_x = torch.arange(W, device=device).view(1, 1, 1, W).expand(B, 1, H, W).float()
        x_proj = grid_x - disp_LR
        grid_y = torch.arange(H, device=device).view(1, 1, H, 1).expand(B, 1, H, W).float()
        
        norm_x = 2.0 * x_proj / (W - 1) - 1.0
        norm_y = 2.0 * grid_y / (H - 1) - 1.0
        # FIX 4: squeeze(1) vor stack (wie im Loss Fix)
        grid = torch.stack((norm_x.squeeze(1), norm_y.squeeze(1)), dim=3)
        
        disp_RL_warped = F.grid_sample(disp_RL, grid, align_corners=False, padding_mode='border')
        lrc_diff = torch.abs(disp_LR - disp_RL_warped)

    # Plotting Helper
    def to_np(t): return t.squeeze().cpu().numpy()
    
    fig, axs = plt.subplots(2, 2, figsize=(12, 8))
    
    # 1. LR Prediction
    im1 = axs[0,0].imshow(to_np(disp_LR), cmap='magma', vmin=0, vmax=192)
    axs[0,0].set_title("LR Prediction")
    fig.colorbar(im1, ax=axs[0,0])
    
    # 2. RL Prediction (Warped to Left)
    im2 = axs[0,1].imshow(to_np(disp_RL_warped), cmap='magma', vmin=0, vmax=192)
    axs[0,1].set_title("RL Prediction (Warped to Left)")
    
    # 3. LRC Differenz
    im3 = axs[1,0].imshow(to_np(lrc_diff), cmap='hot', vmin=0, vmax=10)
    axs[1,0].set_title("LRC Difference (Consistency Check)")
    fig.colorbar(im3, ax=axs[1,0])
    
    # 4. Bild Overlay (Fix für Grayscale)
    left_img = to_np(l)
    if left_img.ndim == 3:
        left_img = left_img.transpose(1, 2, 0)
    axs[1,1].imshow(left_img, cmap='gray' if left_img.ndim == 2 else None)
    axs[1,1].set_title("Left Image")

    for ax in axs.flat: ax.axis('off')
    
    os.makedirs("training_progress", exist_ok=True)
    plt.savefig(f"training_progress/FusedBackbone-Stereo_symmetry_ep{epoch:03d}.png")
    plt.close(fig)


In [ ]:
# --- CELL 4: MAIN V9 (CONFIGURATION & EXECUTION) ---


def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def init_weights_v9(m):
    """
    Spezielle Initialisierung für V9:
    Wir nutzen Kaiming Init für unsere neuen Layer (Refiner, CostVol, Heads), 
    aber lassen den Pre-Trained Backbone in Ruhe!
    """
    if isinstance(m, (nn.Conv2d, nn.Linear)):
        # ReLU-optimiertes Init (Kaiming / He)
        nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
        if m.bias is not None:
            nn.init.constant_(m.bias, 0)
    elif isinstance(m, (nn.BatchNorm2d, nn.GroupNorm)):
        nn.init.constant_(m.weight, 1)
        nn.init.constant_(m.bias, 0)

def main():
    # 1. Reproduzierbarkeit & Cleanup
    set_seed(42)
    gc.collect()
    torch.cuda.empty_cache()
    
    # 2. Modell V9 instanziieren
    print("🏗️ Erstelle StereoNet V9 (MobileNetV3-Large + Mini U-Nets)...")
    # Stelle sicher, dass StereoNet_NPU_V9 vorher definiert wurde
    model = StereoNet_NPU_V9(max_disp=192).to(device)
    
    # 3. Selektive Initialisierung (Smart Init)
    print("🎨 Initialisiere Gewichte (Pre-trained Backbone preserved)...")
    
    # Wir iterieren über die Hauptblöcke des Modells
    for name, module in model.named_children():
        if name == 'backbone':
            print(f"   -> 🔒 Skipping initialization for pretrained: {name}")
            # Der Backbone behält seine ImageNet-Gewichte
        else:
            print(f"   -> 🖌️ Initializing new layers: {name}")
            module.apply(init_weights_v9)
            
    # Anpassung für kleine Batchsizes (BS=6):
    # Setze Momentum von BatchNorm runter, damit der Running Mean nicht zu stark schwankt
    model.apply(lambda m: setattr(m, 'momentum', 0.05) if isinstance(m, nn.BatchNorm2d) else None)

    # 4. Pfad Konfiguration
    # Dein Pfad aus dem vorherigen Setup
    dataset_path = r"/home/slarc/datasets/sceneflow" 
    
    if not os.path.exists(dataset_path):
        print(f"⚠️ KRITISCH: Pfad {dataset_path} nicht gefunden!")
        return # Abbruch, um Fehler zu vermeiden
    
    # 5. Training Starten
    train_strategic_v9(
        model=model,
        base_dir=dataset_path,
        
        # Hyperparameter für V9
        epochs=75,          
        lr_max=2e-4,        # Peak LR für OneCycle
        weight_decay=1e-5,  
        warmup_pct=0.1,      # Prozent der Epochen für Warmup
        grad_clip=1.0,
        # Hardware & Data
        batch_size=6,       # Optimiert für VRAM (MobileNet + U-Nets)
        num_workers=4,
        use_crop=True,       # Training mit Random Crop?
        use_aug=True           # Training mit Color Augmentation?
        )

if __name__ == '__main__':
    main()

    


/tmp/ipykernel_3689937/3716136832.py:174: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


[TRAIN] Dataset: 21818 Bilder (Grayscale + Resize 640x480)
[VAL] Dataset: 4248 Bilder (Grayscale + Resize 640x480)
🚀 TRAINING START | 75 Epochen | BS=16 | LR=2.0e-04
   Strategie: 30 Epochen Geometrie -> Dann Refinement


Ep 1 [INITIAL]:   0%|                                                                                                                  | 0/1364 [00:00<?, ?it/s]/tmp/ipykernel_3689937/3716136832.py:209: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
/home/slarc/miniconda3/envs/stereo_wsl/lib/python3.10/site-packages/torch/optim/lr_scheduler.py:224: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  warnings.warn(
Ep 1 [INITIAL]: 100%|████████████████████████████████| 1364/1364 [07:34<00:00,  3.00it/s, VRAM=8.3G, Loss=5.85, G=5.29, A=0.02, Grad=

📈 Epoche 1 [INITIAL]: Val-EPE: 11.56 px


Ep 2 [INITIAL]: 100%|████████████████████████████████| 1364/1364 [07:31<00:00,  3.02it/s, VRAM=8.3G, Loss=7.21, G=6.72, A=0.02, Grad=15.8, Damp=0.0, LR=4.0e-05]
                                                                                                                                                      

...

Ep 50 [REFINE]: 100%|████████████████████████████████| 1364/1364 [07:40<00:00,  2.96it/s, VRAM=8.3G, Loss=1.37, G=1.32, A=0.01, Grad=25.7, Damp=0.0, LR=6.0e-05]
                                                                                                                                                      

📈 Epoche 50 [REFINE]: Val-EPE: 2.14 px


Ep 51 [REFINE]: 100%|████████████████████████████████| 1364/1364 [07:33<00:00,  3.01it/s, VRAM=8.3G, Loss=1.66, G=1.62, A=0.01, Grad=16.0, Damp=0.0, LR=5.6e-05]
                                                                                                                                                      

📈 Epoche 51 [REFINE]: Val-EPE: 2.21 px


Ep 52 [REFINE]:   3%|█                                 | 44/1364 [00:15<06:54,  3.19it/s, VRAM=8.3G, Loss=1.04, G=0.92, A=0.01, Grad=10.3, Damp=0.0, LR=5.6e-05]

In [ ]:
%abort

In [ ]:
import matplotlib.pyplot as plt
full_dataset = StereoDataset(
        left_dir='/home/slarc/datasets/sceneflow/left',
        right_dir='/home/slarc/datasets/sceneflow/right',
        disp_dir='/home/slarc/datasets/sceneflow/disp',
        training=True
    )
val_ratio = 0.1
val_size = int(len(full_dataset) * val_ratio)
train_size = len(full_dataset) - val_size

train_dataset, val_dataset = torch.utils.data.random_split(
        full_dataset,
        [train_size, val_size],
        generator=torch.Generator().manual_seed(42)
    )

# Ein Bild aus dem Dataset holen
left, right, gt = train_dataset[7491] 

plt.figure(figsize=(15, 5))
plt.subplot(1, 3, 1)
plt.imshow(left.squeeze(), cmap='gray')
plt.title("Eingangsbild (Ist es aufrecht?)")

plt.subplot(1, 3, 2)
plt.imshow(gt.squeeze(), cmap='jet')
plt.title("GT Disparität")

# Check: Wo sind die Gradienten am stärksten?
# Wenn dy > dx, dann ist die Disparität vertikal orientiert!
dy, dx = torch.gradient(gt.squeeze())
plt.subplot(1, 3, 3)
plt.imshow(dx.abs() > dy.abs(), cmap='gray')
plt.title("Weiß = Horizontale Struktur\nSchwarz = Vertikale Struktur")
plt.show()

In [ ]:
# Testen Sie:
feat = make_feature_extractor()
dummy = torch.randn(1, 1, 480, 640)
out = feat(dummy)
print(f"Feature channels: {out.shape[1]}")  # Muss 32 sein!

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
from torchvision import transforms
from PIL import Image

# ---------------------------------------------------------
# Hilfsfunktion: Bild laden und normalisieren
# ---------------------------------------------------------
def load_gray_image(path):
    img = Image.open(path).convert("L")
    t = transforms.ToTensor()
    return t(img).unsqueeze(0).cuda()

# Pfade und Device
left_path  = "/home/slarc/datasets/sceneflow/left/0000006.png"
right_path = "/home/slarc/datasets/sceneflow/right/0000006.png"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

left  = load_gray_image(left_path)
right = load_gray_image(right_path)

# Modell laden (Stelle sicher, dass die Klasse definiert ist)
model = StereoFusionAllMode().to(device)
model.eval()

with torch.no_grad():
    # 1. Fast Mode (Nur LR Pass)
    out_fast = model(left, right, mode="fast")
    # Extraktion aus dem Dictionary-Key "LR"
    disp_fast = out_fast["LR"][0]
    occ_fast  = out_fast["LR"][3]

    # 2. Precise Mode (LR und geflippter RL Pass)
    out_prec = model(left, right, mode="precise")
    disp_prec_LR = out_prec["LR"][0]   # Normaler Pass
    disp_prec_RL = out_prec["RL"][0]   # Symmetrischer RL-Pass (bereits zurückgeflippt!)
    
    # Echo-Analyse: Wo unterscheiden sich LR und RL? (Meist am linken Rand)
    echo_map = torch.abs(disp_prec_LR - disp_prec_RL)

# ---------------------------------------------------------
# Visualisierung: Der "Miststück-Check"
# ---------------------------------------------------------
def show_disp(disp, title, subplot_pos, cmap="magma"):
    plt.subplot(2, 2, subplot_pos)
    disp_np = disp.squeeze().cpu().numpy()
    plt.imshow(disp_np, cmap=cmap, vmin=0, vmax=192)
    plt.colorbar(label="Pixel")
    plt.title(title)
    plt.axis("off")

plt.figure(figsize=(16, 10))

# Oben Links: Fast Mode (Standard)
show_disp(disp_fast, "Fast Mode (LR only)", 1)

# Oben Rechts: Precise Mode LR
show_disp(disp_prec_LR, "Precise Mode (LR Pass)", 2)

# Unten Links: Precise Mode RL (Der Retter für den linken Rand)
show_disp(disp_prec_RL, "Precise Mode (RL Pass - Flipped)", 3)

# Unten Rechts: Echo-Analyse (LRC-Diff)
# Hier siehst du die Fehler am linken Rand leuchten!
show_disp(echo_map, "Echo Analysis (LRC Diff)", 4, cmap="hot")

plt.tight_layout()
plt.show()

# ---------------------------------------------------------
# EPE Check
# ---------------------------------------------------------
# Falls gt_disp.npy nicht existiert, erstellen wir eine Dummy-Maske zum Testen
try:
    gt = np.load("gt_disp.npy")
    gt = torch.tensor(gt, dtype=torch.float32).unsqueeze(0).unsqueeze(0).cuda()
    valid = gt > 0
    
    def calc_epe(pred, gt_val, mask):
        return torch.abs(pred - gt_val)[mask].mean().item()

    epe_fast = calc_epe(disp_fast, gt, valid)
    epe_prec = calc_epe(disp_prec_LR, gt, valid)

    print("-" * 30)
    print(f"EPE Fast Mode   : {epe_fast:.4f} px")
    print(f"EPE Precise Mode: {epe_prec:.4f} px")
    print("-" * 30)
except FileNotFoundError:
    print("GT Datei nicht gefunden. Überspringe EPE Check.")


In [ ]:
import torch
from your_model_file import StereoNetLite_GrabberCore

# 1. Load trained model
model = StereoNetLite_GrabberCore(max_disp=192, num_groups=4, input_size=(480, 640))
model.load_state_dict(torch.load("best_model.pth"))
model.eval()

# 2. Create dummy inputs
dummy_left = torch.randn(1, 1, 480, 640)
dummy_right = torch.randn(1, 1, 480, 640)

# 3. Test forward pass
with torch.no_grad():
    output = model(dummy_left, dummy_right, training=False)
    print(f"Output shape: {output.shape}")  # Should be [1, 1, 480, 640]

# 4. Export to ONNX
torch.onnx.export(
    model,
    (dummy_left, dummy_right),
    "stereo_hailo.onnx",
    export_params=True,
    opset_version=11,
    do_constant_folding=True,
    input_names=['left_image', 'right_image'],
    output_names=['disparity'],
    dynamic_axes={
        'left_image': {0: 'batch_size'},
        'right_image': {0: 'batch_size'},
        'disparity': {0: 'batch_size'}
    }
)

print("✅ ONNX export successful: stereo_hailo.onnx")

# 5. Verify ONNX
import onnx
onnx_model = onnx.load("stereo_hailo.onnx")
onnx.checker.check_model(onnx_model)
print("✅ ONNX model is valid")

# 6. Test ONNX inference
import onnxruntime as ort
session = ort.InferenceSession("stereo_hailo.onnx")
onnx_output = session.run(
    None,
    {'left_image': dummy_left.numpy(), 'right_image': dummy_right.numpy()}
)
print(f"✅ ONNX inference successful, output shape: {onnx_output[0].shape}")

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

# --- 1. Deine vorhandenen Hilfsfunktionen (PFM & Bilder) ---
def read_pfm(file):
    with open(file, "rb") as f:
        header = f.readline().decode('utf-8').rstrip()
        if header != 'Pf': raise Exception('Keine PFM Pf-Datei.')
        dims = f.readline().decode('utf-8').split()
        width, height = int(dims[0]), int(dims[1])
        scale = float(f.readline().decode('utf-8').rstrip())
        endian = '<' if scale < 0 else '>'
        data = np.fromfile(f, endian + 'f')
        data = np.reshape(data, (height, width))
        data = np.flipud(data)
        data[data == np.inf] = 0
        return data.copy()

# --- 2. Die Analyse-Funktion ---
import torch
import torch.nn.functional as F
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

def analyze_checkpoint_pil(checkpoint_path, left_path, right_path, gt_path, row_y=120):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    target_size = (640, 480) # (W, H)
    
    # 1. Modell laden (weights_only=True für Sicherheit)
    model = StereoFusionAllMode().to(device)
    state_dict = torch.load(checkpoint_path, map_location=device, weights_only=True)
    model.load_state_dict(state_dict)
    model.eval()

    # 2. Bilder laden & Resizen (Dein Code)
    left_img_pil = Image.open(left_path).convert('L').resize(target_size, Image.BILINEAR)
    right_img_pil = Image.open(right_path).convert('L').resize(target_size, Image.BILINEAR)
    
    # In Tensor umwandeln [1, 1, 480, 640]
    img_l = torch.from_numpy(np.array(left_img_pil)).float().unsqueeze(0).unsqueeze(0).to(device)
    img_r = torch.from_numpy(np.array(right_img_pil)).float().unsqueeze(0).unsqueeze(0).to(device)

    # 3. Ground Truth laden & Resizen
    gt_orig = read_pfm(gt_path) # Nutzt deine Funktion
    orig_h, orig_w = gt_orig.shape
    
    # WICHTIG: Wenn wir das Bild verkleinern, müssen wir die Disparitätswerte skalieren!
    scale_factor = target_size[0] / orig_w
    gt_rescaled = F.interpolate(torch.from_numpy(gt_orig).unsqueeze(0).unsqueeze(0), 
                                size=(target_size[1], target_size[0]), 
                                mode='nearest').squeeze().numpy()
    gt_rescaled = gt_rescaled * scale_factor # Werte an neue Auflösung anpassen

    # 4. Inferenz
    with torch.no_grad():
        outputs = model(img_l, img_r, mode="precise")
        pred_lr = outputs["LR"][0].cpu().squeeze().numpy()

    # 5. Plotten
    plt.figure(figsize=(15, 6))
    x = np.arange(target_size[0])
    
    plt.plot(x, gt_rescaled[row_y, :], color='black', label='GT (PFM, skaliert)', linewidth=2)
    plt.plot(x, pred_lr[row_y, :], color='red', label='Prediction LR', alpha=0.8)
    
    plt.fill_between(x, gt_rescaled[row_y, :], pred_lr[row_y, :], 
                     where=(np.abs(pred_lr[row_y, :] - gt_rescaled[row_y, :]) > 3),
                     color='red', alpha=0.1, label='Fehler > 3px')

    plt.title(f"Profil-Check Zeile {row_y} (Skalierung: {orig_w} -> {target_size[0]})")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()

# Aufruf

# ================================================================
# --- 3. DER FUNKTIONSAUFRUF (HIER PASSIERT ES) ---
# ================================================================

if __name__ == "__main__":
    # Pfade anpassen!
    MY_CHECKPOINT = "FusedBackbone-Stereo_5.pth"
    TEST_L = "/home/slarc/datasets/sceneflow/left/0000052.png"
    TEST_R = "/home/slarc/datasets/sceneflow/right/0000052.png"
    TEST_GT = "/home/slarc/datasets/sceneflow/disp/0000052.pfm"

    # Aufruf für die Problem-Zone (Stuhlbein-Echo oben links)
    analyze_checkpoint(
        checkpoint_path=MY_CHECKPOINT,
        left_path=TEST_L,
        right_path=TEST_R,
        gt_path=TEST_GT,
        row_y=120  # Wähle die Zeile, in der das Stuhlbein im Bild sitzt
    )

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

import os
path = '/mnt/c/temp/sceneflow/disp/' # Update this
print(f"Directory exists: {os.path.exists(path)}")
print(f"Files in directory: {os.listdir(path)[:5]}") # Shows first 5 files

# 1. Function Call
# Replace 'path_to_your_file.pfm' with your actual file path
file_path = '/mnt/c/temp/FlyingThings3D_subset_disparity.tar/FlyingThings3D_subset_disparity/FlyingThings3D_subset/val/disparity/right/0001000.pfm'
try:
    disparity_map = read_pfm(file_path)
    
    # 2. Visualization
    plt.figure(figsize=(12, 6))
    
    # Use 'magma' or 'plasma' for depth/disparity; it's easier on the eyes
    img = plt.imshow(disparity_map, cmap='magma')
    
    plt.title(f"Stereo Ground Truth Disparity\nResolution: {disparity_map.shape[1]}x{disparity_map.shape[0]}")
    plt.colorbar(img, label='Disparity (pixels)')
    plt.axis('off') # Hide axes for a cleaner look
    
    plt.show()
except FileNotFoundError:
    print(f"Error: The file at {file_path} was not found.")
except Exception as e:
    print(f"An error occurred: {e}")

file_path = '/mnt/c/temp/FlyingThings3D_subset_disparity.tar/FlyingThings3D_subset_disparity/FlyingThings3D_subset/val/disparity/left/0001000.pfm'
try:
    disparity_map = read_pfm(file_path)
    
    # 2. Visualization
    plt.figure(figsize=(12, 6))
    
    # Use 'magma' or 'plasma' for depth/disparity; it's easier on the eyes
    img = plt.imshow(disparity_map, cmap='magma')
    
    plt.title(f"Stereo Ground Truth Disparity\nResolution: {disparity_map.shape[1]}x{disparity_map.shape[0]}")
    plt.colorbar(img, label='Disparity (pixels)')
    plt.axis('off') # Hide axes for a cleaner look
    
    plt.show()

except FileNotFoundError:
    print(f"Error: The file at {file_path} was not found.")
except Exception as e:
    print(f"An error occurred: {e}")

In [ ]:
import os
path = '/mnt/c/temp/sceneflow/disp/' # Update this
print(f"Directory exists: {os.path.exists(path)}")
print(f"Files in directory: {os.listdir(path)[:5]}") # Shows first 5 files